# Run 6 — Analysis & Export

Loads saved checkpoints and result JSONs from run6. No training.

**Prerequisites**: run6 must be complete and results synced to Drive.
- `results/ablations/run6/_final_config.json`
- `results/ablations/run6/*.json`
- `models/run6/{winner_name}.pt`

| Section | What it does |
|---|---|
| §8a | Confusion matrix (needs GPU + checkpoint) |
| §8b | Class distribution (reads dataset only) |
| §8c | Shuttle coverage (reads dataset only) |
| §8d | Overfitting curves (reads JSONs only) |
| §8e | Diagnostic summary |
| §10 | Per-shot inference export → `shot_type_predictions.json` |

In [ ]:
import os, sys, json
from pathlib import Path

try:
    import google.colab; IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT   = Path('/content/drive/MyDrive/Baddiev2')
    PROJECT_PATH = Path('/content/Baddiev2')
    import zipfile
    if not (PROJECT_PATH / 'src').exists():
        with zipfile.ZipFile(DRIVE_ROOT / 'baddiev2_colab.zip') as z:
            z.extractall(PROJECT_PATH)
    sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)
else:
    PROJECT_PATH = Path('..').resolve()
    if not (PROJECT_PATH / 'src').exists():
        PROJECT_PATH = Path('.').resolve()
    DRIVE_ROOT = PROJECT_PATH
    if str(PROJECT_PATH) not in sys.path:
        sys.path.insert(0, str(PROJECT_PATH))
    os.chdir(PROJECT_PATH)

import src.config as _cfg

if IN_COLAB:
    _cfg.SS_SKELETONS_GDINO = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_skeletons_gdino'
    _cfg.SS_SHUTTLES        = DRIVE_ROOT / 'datasets_preprocessing' / 'shuttleset_shuttles'
    _cfg.SS_SPLIT_JSON      = PROJECT_PATH / 'datasets_preprocessing' / 'shuttleset_split.json'
    _cfg.MODELS_DIR         = DRIVE_ROOT / 'models'
    _cfg.RESULTS_DIR        = DRIVE_ROOT / 'results'

print(f'Project: {PROJECT_PATH}')
print(f'Colab: {IN_COLAB}')

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
from torch.utils.data import DataLoader, Dataset as _Dataset
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

from src.config import (
    FEATURE_DIMS, FEATURE_DIMS_WITH_HITTER, BONE_CHANNELS, COCO_SKELETON,
    SS_SKELETONS_GDINO, SS_SHUTTLES, SS_SPLIT_JSON,
    SHOT_TYPES, NUM_SHOT_TYPES, NUM_NODES, NUM_JOINTS, MODELS_DIR, RESULTS_DIR,
)
from src.data.graph_builder import GraphBuilder
from src.data.dataset import ShuttleSetDataset
from src.models.stgcn_model import STGCN
from src.models.shuttle_cross_attn import ShuttleCrossAttention

device = torch.device(
    'cuda' if torch.cuda.is_available() else
    'mps'  if torch.backends.mps.is_available() else
    'cpu'
)

N_CLASSES     = NUM_SHOT_TYPES
SHOT_WINDOW   = 32
BATCH_SIZE    = 64
ABLATION_DIR  = RESULTS_DIR / 'ablations' / 'run6'
MODELS_DIR_R6 = MODELS_DIR  / 'run6'

# Data directories — use local SSD copy on Colab if available, else Drive
if IN_COLAB:
    local_skel  = Path('/content/local_skel')
    local_shutt = Path('/content/local_shutt')
    import shutil
    if not local_skel.exists():
        print('Copying skeletons to local SSD...')
        shutil.copytree(_cfg.SS_SKELETONS_GDINO, local_skel)
    if not local_shutt.exists():
        print('Copying shuttles to local SSD...')
        shutil.copytree(_cfg.SS_SHUTTLES, local_shutt)
    SKEL_DIR  = local_skel
    SHUTT_DIR = local_shutt
else:
    SKEL_DIR  = _cfg.SS_SKELETONS_GDINO
    SHUTT_DIR = _cfg.SS_SHUTTLES

print(f'Device: {device}')
print(f'N_CLASSES: {N_CLASSES}')
print(f'Ablation dir: {ABLATION_DIR}')

In [ ]:
# ── Helper functions ──────────────────────────────────────────────────────

class SinglePlayerWrapper(_Dataset):
    """Extract only the hitter's 17 joints from a dual-player dataset."""
    def __init__(self, ds):
        self.ds = ds
        self.samples = ds.samples
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        item = self.ds[idx]
        x, label = item[0], item[1]
        info   = self.ds.samples[idx]
        hitter = info.get('hitter', 'top') if isinstance(info, dict) else 'top'
        x = x[:, :, NUM_JOINTS:] if hitter == 'bottom' else x[:, :, :NUM_JOINTS]
        return x, label


class SinglePlayerCrossAttnWrapper(_Dataset):
    """Hitter 17 joints + preserved shuttle tensor. For cross-attn + single player."""
    def __init__(self, ds):
        self.ds = ds
        self.samples = ds.samples
    def __len__(self): return len(self.ds)
    def __getitem__(self, idx):
        item     = self.ds[idx]
        x, label = item[0], item[1]
        shuttle  = item[2] if len(item) == 3 else torch.zeros(2, x.shape[1])
        info   = self.ds.samples[idx]
        hitter = info.get('hitter', 'top') if isinstance(info, dict) else 'top'
        x = x[:, :, NUM_JOINTS:] if hitter == 'bottom' else x[:, :, :NUM_JOINTS]
        return x, label, shuttle


def collate_fn(batch):
    xs, labels = zip(*[(b[0], b[1]) for b in batch])
    return torch.stack(xs), torch.tensor(labels, dtype=torch.long)


def collate_fn_shuttle(batch):
    xs, labels, shuttles = [], [], []
    for item in batch:
        xs.append(item[0]); labels.append(item[1])
        shuttles.append(item[2] if len(item) == 3 else torch.zeros(2, SHOT_WINDOW))
    return torch.stack(xs), torch.tensor(labels, dtype=torch.long), torch.stack(shuttles)


def build_encoder(in_channels, num_nodes=NUM_NODES, pooling='mean'):
    single = (num_nodes == NUM_JOINTS)
    adj = GraphBuilder(use_inter_player=not single, single_player=single).build_adjacency().to(device)
    return STGCN(in_channels=in_channels, num_nodes=num_nodes, adjacency=adj,
                 num_layers=9, base_channels=64, embedding_dim=256,
                 temporal_kernel=9, dropout=0.3, pooling=pooling).to(device)


def compute_class_weights(dataset):
    labels  = [s.get('shot_type_idx') for s in dataset.samples
               if s.get('shot_type_idx') is not None]
    counts  = Counter(labels)
    total   = sum(counts.values())
    weights = torch.ones(N_CLASSES, dtype=torch.float32)
    for cls_id, cnt in counts.items():
        if cls_id < N_CLASSES:
            weights[cls_id] = total / (len(counts) * cnt)
    return weights


def evaluate(encoder, head, ds, cross_attn=None):
    cfn    = collate_fn_shuttle if cross_attn else collate_fn
    loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=0, collate_fn=cfn)
    encoder.eval(); head.eval()
    if cross_attn: cross_attn.eval()
    all_logits, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            xb, yb = batch[0], batch[1]; valid = yb >= 0
            if not valid.any(): continue
            emb = encoder(xb[valid].to(device))
            if cross_attn: emb = cross_attn(emb, batch[2][valid].to(device))
            all_logits.append(head(emb).cpu()); all_labels.append(yb[valid])
    if not all_logits: return 0.0, 0.0, np.array([]), np.array([]), {}
    logits = torch.cat(all_logits); y_true = torch.cat(all_labels).numpy()
    y_pred = logits.argmax(1).numpy()
    topk = {f'top{k}_acc': logits.topk(k,1).indices.eq(
                torch.tensor(y_true).unsqueeze(1)).any(1).float().mean().item()
            for k in [3, 5] if logits.shape[1] >= k}
    return (f1_score(y_true, y_pred, average='macro', zero_division=0),
            accuracy_score(y_true, y_pred), y_true, y_pred, topk)


print('Helpers defined.')

In [ ]:
# ── Load final config + splits ────────────────────────────────────────────
with open(ABLATION_DIR / '_final_config.json') as f:
    fc = json.load(f)
with open(SS_SPLIT_JSON) as f:
    splits = json.load(f)

VAL_MATCH_NAMES = [
    'NG_Ka_Long_Angus_Jonatan_CHRISTIE_Malaysia_Masters_2020_QuarterFinals',
    'Anders_Antonsen_Sameer_Verma_TOYOTA_THAILAND_OPEN_2021_QuarterFinals',
]
TRAIN_MATCHES = set(splits['train']) - set(VAL_MATCH_NAMES)
VAL_MATCHES   = set(VAL_MATCH_NAMES)
TEST_MATCHES  = set(splits['held_out'])
ALL_MATCHES   = set(splits['train']) | set(splits['held_out'])

print(f'Winner: {fc["winner_name"]}')
print(f'Config: {fc}')
print(f'Train: {len(TRAIN_MATCHES)}  Val: {len(VAL_MATCHES)}  Test: {len(TEST_MATCHES)}')

## §8 — Diagnostic Analysis

Run cells in order — §8e depends on variables computed in §8a–§8d.

In [ ]:
# ── 8a — Confusion Matrix ─────────────────────────────────────────────────
# Loads checkpoint and runs inference on test set.

def _make_test_ds(fc, match_set):
    ds = ShuttleSetDataset(
        skeleton_dir=SKEL_DIR, shot_window=SHOT_WINDOW,
        feature_layer=fc['feature_layer'], load_shot_types=True, split=None,
        use_shuttle=fc['use_shuttle'], shuttle_dir=SHUTT_DIR if fc['use_shuttle'] else None,
        variable_window=fc['variable_window'], shuttle_fusion=fc['shuttle_fusion'],
        use_hitter=fc['use_hitter'], use_bones=fc['use_bones'], use_bbox_norm=fc['use_bbox_norm'],
    )
    ds.samples = [s for s in ds.samples
                  if isinstance(s, dict) and Path(s.get('skel_dir', '')).name in match_set]
    if fc['single_player']:
        if fc['use_shuttle'] and fc['shuttle_fusion'] == 'cross_attn':
            return SinglePlayerCrossAttnWrapper(ds)
        return SinglePlayerWrapper(ds)
    return ds


def _load_model(fc):
    ckpt = torch.load(MODELS_DIR_R6 / f'{fc["winner_name"]}.pt',
                      map_location=device, weights_only=True)
    enc  = build_encoder(fc['in_channels'], num_nodes=fc['num_nodes'], pooling=fc['pooling'])
    enc.load_state_dict(ckpt['enc']);  enc.eval()
    head = nn.Linear(256, N_CLASSES).to(device)
    head.load_state_dict(ckpt['head']); head.eval()
    ca = None
    if fc['use_shuttle'] and fc['shuttle_fusion'] == 'cross_attn':
        ca = ShuttleCrossAttention(d_skel=256, d_shuttle=128, nhead=4).to(device)
        ca.load_state_dict(ckpt['ca']); ca.eval()
    return enc, head, ca


test_ds = _make_test_ds(fc, TEST_MATCHES)
enc, head, ca = _load_model(fc)
print(f'Test samples: {len(test_ds)}')

_macro_f1, _accuracy, y_true, y_pred, topk = evaluate(enc, head, test_ds, ca)
for k in [1, 3, 5]:
    if k == 1:
        print(f'Accuracy:    {_accuracy:.4f}')
        print(f'Macro-F1:    {_macro_f1:.4f}')
    else:
        print(f'Top-{k} Acc: {topk.get(f"top{k}_acc", 0):.4f}')

_present = sorted(set(y_true) | set(y_pred))
_lnames  = [SHOT_TYPES[i] if i < len(SHOT_TYPES) else f'cls_{i}' for i in _present]
_cm      = confusion_matrix(y_true, y_pred, labels=_present)
_cm_norm = _cm.astype(float) / _cm.sum(axis=1, keepdims=True)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(22, 9))
for ax, data, title, fmt, vmax in [
    (ax1, _cm,      f'Counts — {fc["winner_name"]}', 'd',   None),
    (ax2, _cm_norm, f'Row-norm — {fc["winner_name"]}', '.2f', 1),
]:
    kw = {'vmin': 0, 'vmax': vmax} if vmax else {}
    im = ax.imshow(data, interpolation='nearest', cmap='Blues', **kw)
    ax.set_title(title, fontsize=11)
    ax.set_xticks(range(len(_lnames))); ax.set_yticks(range(len(_lnames)))
    ax.set_xticklabels(_lnames, rotation=45, ha='right', fontsize=8)
    ax.set_yticklabels(_lnames, fontsize=8)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    thresh = data.max() * 0.5 if vmax is None else 0.5
    for i in range(len(_lnames)):
        for j in range(len(_lnames)):
            val = data[i, j]
            if (val > 0.005 if isinstance(val, float) else val > 0):
                ax.text(j, i, format(val, fmt), ha='center', va='center',
                        fontsize=6, color='white' if val > thresh else 'black')
    fig.colorbar(im, ax=ax, shrink=0.8)
plt.tight_layout()
out = ABLATION_DIR / f'confusion_matrix_{fc["winner_name"]}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

_pairs = [(int(_cm[i,j]), float(_cm_norm[i,j]), _lnames[i], _lnames[j])
          for i in range(len(_lnames)) for j in range(len(_lnames))
          if i != j and _cm[i,j] > 0]
_pairs.sort(reverse=True)
print('\nTop 15 confusion pairs:')
for cnt, frac, tc, pc in _pairs[:15]:
    print(f'  {tc:<24} → {pc:<24} : {cnt:4d} ({frac:.0%} of true {tc})')

In [ ]:
# ── 8b — Class Distribution ───────────────────────────────────────────────

def _label_counts(match_set, label):
    ds = ShuttleSetDataset(skeleton_dir=SKEL_DIR, shot_window=SHOT_WINDOW,
                           feature_layer='L2', load_shot_types=True, split=None)
    ds.samples = [s for s in ds.samples
                  if isinstance(s, dict) and Path(s.get('skel_dir', '')).name in match_set]
    labels = [s.get('shot_type_idx') for s in ds.samples if s.get('shot_type_idx') is not None]
    print(f'{label}: {len(labels)} labeled samples')
    return Counter(labels)

tr_cnt = _label_counts(TRAIN_MATCHES, 'Train')
va_cnt = _label_counts(VAL_MATCHES,   'Val')
te_cnt = _label_counts(TEST_MATCHES,  'Test')
all_cls = sorted(set(tr_cnt) | set(va_cnt) | set(te_cnt))
tot_tr  = sum(tr_cnt.values())

print(f'\n{"ID":<4} {"Shot Type":<26} {"Train":>7} {"Tr%":>6} {"Val":>6} {"Test":>6} {"Total":>7}')
print('-' * 65)
for c in all_cls:
    nm = SHOT_TYPES[c] if c < len(SHOT_TYPES) else f'cls_{c}'
    tr = tr_cnt.get(c, 0); va = va_cnt.get(c, 0); te = te_cnt.get(c, 0)
    print(f'{c:<4} {nm:<26} {tr:>7} {tr/tot_tr*100:>5.1f}% {va:>6} {te:>6} {tr+va+te:>7}')
print(f'     {"TOTAL":<26} {tot_tr:>7} {"100%":>6} {sum(va_cnt.values()):>6} '
      f'{sum(te_cnt.values()):>6} {tot_tr+sum(va_cnt.values())+sum(te_cnt.values()):>7}')
print(f'\nImbalance: {max(tr_cnt.values())}/{min(tr_cnt.values())} = '
      f'{max(tr_cnt.values())/min(tr_cnt.values()):.1f}x')

fig, ax = plt.subplots(figsize=(14, 5))
_names = [SHOT_TYPES[c] if c < len(SHOT_TYPES) else f'cls_{c}' for c in all_cls]
_x = np.arange(len(all_cls)); _w = 0.28
ax.bar(_x-_w, [tr_cnt.get(c,0) for c in all_cls], _w, label='Train', color='#3498db')
ax.bar(_x,    [va_cnt.get(c,0) for c in all_cls], _w, label='Val',   color='#2ecc71')
ax.bar(_x+_w, [te_cnt.get(c,0) for c in all_cls], _w, label='Test',  color='#e74c3c')
ax.set_xticks(_x); ax.set_xticklabels(_names, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Sample Count'); ax.set_title('Class Distribution: Train / Val / Test')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
out = ABLATION_DIR / 'class_distribution.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── 8c — Shuttle Coverage ─────────────────────────────────────────────────

ds_sh = ShuttleSetDataset(
    skeleton_dir=SKEL_DIR, shot_window=SHOT_WINDOW, feature_layer='L2',
    load_shot_types=True, split=None,
    use_shuttle=True, shuttle_dir=SHUTT_DIR, shuttle_fusion='cross_attn',
)
ds_sh.samples = [s for s in ds_sh.samples
                 if isinstance(s, dict) and Path(s.get('skel_dir','')).name in ALL_MATCHES]

_n_check = min(len(ds_sh), 2000)
_rng = np.random.default_rng(42)
_idx = _rng.choice(len(ds_sh), size=_n_check, replace=False)
print(f'Checking {_n_check} of {len(ds_sh)} samples...')

_n_has = 0; _dens = []
for i in _idx:
    item = ds_sh[int(i)]
    if len(item) >= 3:
        sh = item[2].numpy() if hasattr(item[2], 'numpy') else item[2]
        has = np.any(sh[:2] != 0)
        _n_has += int(has)
        _dens.append(np.any(sh[:2] != 0, axis=0).mean())

_pct = _n_has / _n_check * 100
print(f'With shuttle: {_n_has}/{_n_check} ({_pct:.1f}%)')
_pos = np.array(_dens); _pos = _pos[_pos > 0]
if len(_pos) > 0:
    print(f'Frame density (samples with data): mean={_pos.mean():.1%}  median={np.median(_pos):.1%}')

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(np.array(_dens), bins=20, color='#3498db', edgecolor='white')
ax.set_xlabel('Fraction of frames with shuttle data'); ax.set_ylabel('Count')
ax.set_title(f'Shuttle Trajectory Density  (n={_n_check})')
ax.axvline(x=0.01, color='red', ls='--', alpha=0.5, label=f'No data: {100-_pct:.0f}%')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = ABLATION_DIR / 'shuttle_coverage.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── 8d — Overfitting Analysis ─────────────────────────────────────────────
# Reads JSON history only — no model or dataset needed.

all_res = {}
for p in sorted(ABLATION_DIR.glob('*.json')):
    if p.stem.startswith('_'): continue
    try: all_res[p.stem] = json.load(open(p))
    except: pass
print(f'Loaded {len(all_res)} result files')

print(f'\n{"Name":<30} {"Ep":>4} {"FinalLoss":>10} {"BestValF1":>10} {"TestF1":>8} {"Gap":>8}')
print('-' * 78)
for nm in sorted(all_res):
    r = all_res[nm]; h = r.get('history', {})
    fl  = h['train_loss'][-1] if h.get('train_loss') else None
    bv  = r.get('best_val_f1'); tf = r.get('macro_f1')
    gap = bv - tf if bv is not None and tf is not None else None
    print(f'{nm:<30} {r["stopped_epoch"]:>4} '
          f'{"%10.4f" % fl if fl is not None else "%11s" % "N/A"} '
          f'{"%10.4f" % bv if bv is not None else "%11s" % "N/A"} '
          f'{"%8.4f" % tf if tf is not None else "%9s" % "N/A"} '
          f'{"%+8.4f" % gap if gap is not None else "N/A"}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
_colors = plt.cm.tab10(np.linspace(0, 1, max(len(all_res), 1)))
for (nm, r), c in zip(sorted(all_res.items()), _colors):
    h = r.get('history', {})
    if h.get('train_loss'):
        ax1.plot(range(1, len(h['train_loss'])+1), h['train_loss'], label=nm, color=c, lw=1.2)
    if h.get('val_f1'):
        ax2.plot(range(1, len(h['val_f1'])+1), h['val_f1'], label=nm, color=c, lw=1.2)
ax1.set(xlabel='Epoch', ylabel='Train Loss', title='Train Loss (all variants)')
ax2.set(xlabel='Epoch', ylabel='Val Macro-F1', title='Val F1 (all variants)')
for ax in (ax1, ax2): ax.legend(fontsize=5); ax.grid(True, alpha=0.3)
plt.tight_layout()
out = ABLATION_DIR / 'overfitting_analysis.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print(f'Saved: {out}')

In [ ]:
# ── 8e — Diagnostic Summary ───────────────────────────────────────────────
# Depends on: _pairs (8a), tr_cnt (8b), _pct (8c), all_res (8d)

best_res = json.load(open(ABLATION_DIR / f'{fc["winner_name"]}.json'))
print('=' * 70)
print('DIAGNOSTIC SUMMARY')
print('=' * 70)
print(f'\n1. BEST CONFIG: {fc["winner_name"]}')
print(f'   Test Macro-F1: {best_res["macro_f1"]:.4f}')
print(f'   Test Accuracy: {best_res["accuracy"]:.4f}')
print(f'   Top-3 Acc:     {best_res.get("top3_acc", "N/A")}')
print(f'   Top-5 Acc:     {best_res.get("top5_acc", "N/A")}')
print(f'\n2. CONFIG FLAGS:')
for k, v in fc.items():
    print(f'   {k}: {v}')
print(f'\n3. CLASS IMBALANCE:')
print(f'   Largest: {max(tr_cnt.values())}  Smallest: {min(tr_cnt.values())}  '
      f'Ratio: {max(tr_cnt.values())/min(tr_cnt.values()):.1f}x')
print(f'\n4. SHUTTLE COVERAGE: {_pct:.1f}% of samples have trajectory data')
print(f'\n5. TOP 5 CONFUSION PAIRS:')
for cnt, frac, tc, pc in _pairs[:5]:
    print(f'   {tc} → {pc}: {cnt} ({frac:.0%})')
print(f'\n6. WORST CLASSES BY F1:')
_pc   = best_res.get('per_class', {})
_cf1s = [(v.get('f1-score', 0), v.get('support', 0), k) for k, v in _pc.items()
         if k not in ('accuracy', 'macro avg', 'weighted avg')]
for f1, sup, nm in sorted(_cf1s)[:5]:
    print(f'   {nm:<24} F1={f1:.2f}  (n={sup})')
_h = best_res.get('history', {})
if _h.get('train_loss'):
    bv = best_res['best_val_f1']; tf = best_res['macro_f1']
    print(f'\n7. OVERFITTING:')
    print(f'   Final train loss: {_h["train_loss"][-1]:.4f}')
    print(f'   Best val F1:      {bv:.4f}')
    print(f'   Test F1:          {tf:.4f}')
    print(f'   Gap:              {bv-tf:+.4f} ({"OK" if abs(bv-tf)<0.03 else "notable"})')
print(f'\n8. EVALUATION CAVEAT:')
print(f'   Test set = 2 matches ({len(y_true)} shots). Differences < 0.02 F1 may not be reliable.')
print(f'   Run §9 cross-validation (in run6 notebook) for robust estimates.')

## §10 — Per-Shot Inference Export

Runs the final model on all matches and writes `results/shot_type_predictions.json`.
This file is served by `badminton_server.py` for the UI.

Re-run this cell whenever the checkpoint changes.

In [ ]:
# ── §10 — Per-Shot Inference Export ──────────────────────────────────────

ds_ex = ShuttleSetDataset(
    skeleton_dir=SKEL_DIR, shot_window=SHOT_WINDOW,
    feature_layer=fc['feature_layer'], load_shot_types=True, split=None,
    use_shuttle=fc['use_shuttle'], shuttle_dir=SHUTT_DIR if fc['use_shuttle'] else None,
    variable_window=fc['variable_window'], shuttle_fusion=fc['shuttle_fusion'],
    use_hitter=fc['use_hitter'], use_bones=fc['use_bones'], use_bbox_norm=fc['use_bbox_norm'],
)
ds_ex.samples = [
    s for s in ds_ex.samples
    if isinstance(s, dict) and s.get('shot_type_idx') is not None
    and Path(s.get('skel_dir', '')).name in ALL_MATCHES
]
if fc['single_player']:
    if fc['use_shuttle'] and fc['shuttle_fusion'] == 'cross_attn':
        ds_ex = SinglePlayerCrossAttnWrapper(ds_ex)
    else:
        ds_ex = SinglePlayerWrapper(ds_ex)
print(f'Exporting {len(ds_ex)} labelled samples with {fc["winner_name"]}...')

# Reuse already-loaded model from §8a (or reload if running standalone)
try:
    enc, head, ca
except NameError:
    enc, head, ca = _load_model(fc)

preds = []
with torch.no_grad():
    for idx in range(len(ds_ex)):
        info = (ds_ex.ds.samples if hasattr(ds_ex, 'ds') else ds_ex.samples)[idx]
        item = ds_ex[idx]
        x = item[0].unsqueeze(0).to(device); label = int(item[1])
        if label < 0: continue
        emb = enc(x)
        if ca is not None and len(item) >= 3:
            emb = ca(emb, item[2].unsqueeze(0).to(device))
        logits = head(emb)
        probs  = F.softmax(logits, 1).squeeze(0).cpu().numpy()
        pred_i = int(logits.argmax(1).item())
        top5   = np.argsort(probs)[-5:][::-1]
        match_name = Path(info.get('skel_dir', '')).name
        preds.append({
            'match':      match_name,
            'rally':      info.get('rally_key', ''),
            'ball_round': int(info.get('ball_round', info.get('shot_idx', 0))),
            'frame_num':  int(info.get('frame_num', 0)),
            'true':       SHOT_TYPES[label]  if label  < len(SHOT_TYPES) else f'cls_{label}',
            'pred':       SHOT_TYPES[pred_i] if pred_i < len(SHOT_TYPES) else f'cls_{pred_i}',
            'correct':    label == pred_i,
            'split':      'test' if match_name in splits['held_out'] else 'train',
            'probs':      {SHOT_TYPES[pi] if pi < len(SHOT_TYPES) else f'cls_{pi}':
                           round(float(probs[pi]), 4) for pi in top5},
        })
        if (idx + 1) % 500 == 0:
            print(f'  {idx+1}/{len(ds_ex)}...')

out = RESULTS_DIR / 'shot_type_predictions.json'
with open(out, 'w') as f:
    json.dump(preds, f, indent=2)

_n_ok   = sum(1 for p in preds if p['correct'])
_n_test = sum(1 for p in preds if p['split'] == 'test')
_n_tok  = sum(1 for p in preds if p['split'] == 'test' and p['correct'])
print(f'\nExported {len(preds)} predictions → {out}')
print(f'  Overall: {_n_ok}/{len(preds)} = {_n_ok/len(preds):.4f}')
print(f'  Test:    {_n_tok}/{_n_test} = {_n_tok/max(_n_test,1):.4f}')
print(f'  Train:   {(_n_ok-_n_tok)}/{len(preds)-_n_test} = '
      f'{(_n_ok-_n_tok)/max(len(preds)-_n_test,1):.4f}')